In [1]:
!pip install openpyxl

    PyYAML (>=5.1.*)
            ~~~~~~^

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
tf.config.run_functions_eagerly(True)

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Embedding, LSTM, SimpleRNN, GRU, Dense, TimeDistributed, Dropout, Bidirectional, Input
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from collections import defaultdict

import os

2025-05-07 07:47:07.705668: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-07 07:47:07.840687: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2025-05-07 07:47:07.840722: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2025-05-07 07:47:08.544167: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could 

In [3]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 0


2025-05-07 07:47:09.524971: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2025-05-07 07:47:09.525174: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2025-05-07 07:47:09.525325: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/v

In [4]:
print(tf.test.is_gpu_available())  # Deprecated in TF 2.1
print(tf.config.experimental.list_physical_devices('GPU'))  # Works for TF 2.x

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
False
[]


2025-05-07 07:47:09.542843: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-07 07:47:09.773499: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1934] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [5]:
# Force TensorFlow to use GPU only
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.set_visible_devices(physical_devices[0], 'GPU')

In [6]:
# Allow memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print(e)  # Memory growth must be set at program startup

In [7]:
## Use this for 1L sentences
folder_path = 'test_data'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [8]:
if "kon_agriculture_set1.txt" in txt_files:
    txt_files.remove("kon_agriculture_set1.txt")

print(len(txt_files))

99


In [9]:
taggedData = []
for file in txt_files:
    with open("test_data/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData[:2]

['अस्वस्थतायेचीं\\N_NN कायम\\JJ प्रतिकां\\N_NN .\\RD_PUNC',
 'सगळ्या\\QT_QTF पयल्यान\\PR_PRP प्रशांतीची\\N_NNP उदका\\N_NN रेशा\\N_NN स्पश्ट\\RB जावन\\V_VM_VNF येता\\V_VAUX_VF .\\RD_PUNC']

In [10]:
allTaggedData = list(set(allTaggedData))
len(allTaggedData)

98986

In [11]:
sentences = [sentence.strip().split() for sentence in allTaggedData]

In [12]:
# Extract words and tags
word_tag_pairs = [[token.rsplit("\\", 1) for token in sentence] for sentence in sentences]

In [13]:
# Create word and tag vocabularies
word_freq = defaultdict(int)
tag_freq = defaultdict(int)

In [14]:
filtered_sentences = []

for sentence in word_tag_pairs:
    filtered_sentence = [item for item in sentence if isinstance(item, (list, tuple)) and len(item) == 2]  # Keep only valid pairs
    
    if filtered_sentence:  # Add to filtered list if not empty
        filtered_sentences.append(filtered_sentence)

    for word, tag in filtered_sentence:
        word_freq[word] += 1
        tag_freq[tag] += 1

## Replace original list if needed
word_tag_pairs = filtered_sentences 

In [15]:
print(len(word_freq))
print(len(tag_freq))

136207
80


In [16]:
# Create word and tag mappings
word2idx = {word: idx + 2 for idx, word in enumerate(word_freq.keys())}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {idx: word for word, idx in word2idx.items()}

In [17]:
tag2idx = {tag: idx for idx, tag in enumerate(tag_freq.keys())}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [18]:
# Convert sentences into index sequences
X = [[word2idx.get(word, 1) for word, tag in sentence] for sentence in word_tag_pairs]
y = [[tag2idx[tag] for word, tag in sentence] for sentence in word_tag_pairs]

In [19]:
# Pad sequences
max_len = max(len(sentence) for sentence in X)
X_padded = pad_sequences(X, maxlen=max_len, padding="post")
y_padded = pad_sequences(y, maxlen=max_len, padding="post")
y_categorical = [to_categorical(tag_seq, num_classes=len(tag2idx)) for tag_seq in y_padded]
y_categorical = np.array(y_categorical)

In [20]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_categorical, test_size=0.05, random_state=42)

In [21]:
# Build Bidirectional LSTM Model
input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=len(word2idx), output_dim=128, mask_zero=True)(input_layer)
#lstm_layer = LSTM(units=256, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)(embedding_layer)
#bi_lstm_layer1 = Bidirectional(LSTM(units=256, return_sequences=True, dropout=0.3))(embedding_layer)
#bi_rnn_layer1 = SimpleRNN(units=256, return_sequences=True, dropout=0.3)(embedding_layer)
#bi_rnn_layer1 = Bidirectional(SimpleRNN(units=256, return_sequences=True, dropout=0.3))(embedding_layer)
#bi_gru_layer1 = GRU(units=256, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(embedding_layer)
bi_gru_layer1 = Bidirectional(GRU(units=256, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(embedding_layer)
output_layer = TimeDistributed(Dense(len(tag2idx), activation="softmax"))(bi_gru_layer1)

In [22]:
model = Model(input_layer, output_layer)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [23]:
# If using a custom function
@tf.function
def train_model():
    model.fit(X_train, y_train, batch_size=128, epochs=4, validation_data=(X_test, y_test))

train_model()  

Epoch 1/4


2025-05-07 07:47:34.514607: W tensorflow/tsl/framework/cpu_allocator_impl.cc:82] Allocation of 4019914240 exceeds 10% of free system memory.
/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


733/733 [==============================] - 5145s 7s/step - loss: 0.7213 - accuracy: 0.8015 - val_loss: 0.3729 - val_accuracy: 0.8796
Epoch 2/4
733/733 [==============================] - 5139s 7s/step - loss: 0.2877 - accuracy: 0.9040 - val_loss: 0.3569 - val_accuracy: 0.8845
Epoch 3/4
733/733 [==============================] - 5137s 7s/step - loss: 0.2309 - accuracy: 0.9205 - val_loss: 0.3589 - val_accuracy: 0.8867
Epoch 4/4
733/733 [==============================] - 5120s 7s/step - loss: 0.1947 - accuracy: 0.9337 - val_loss: 0.3739 - val_accuracy: 0.8866


In [24]:
# Save the model
model.save("bigru_pos_tagger.h5")

In [25]:
import dill
with open("word2idx_bigru.pkl", "wb") as f:
    dill.dump(word2idx, f)


with open("idx2tag_bigru.pkl", "wb") as f:
    dill.dump(idx2tag, f)

In [26]:
from keras.models import load_model
loaded_model = load_model("bigru_pos_tagger.h5")

In [27]:
def pos_tag_sentence(sentence):
    tokens = sentence.split()
    token_ids = [word2idx.get(token, 1) for token in tokens]
    token_ids_padded = pad_sequences([token_ids], maxlen=max_len, padding="post")
    predictions = loaded_model.predict(token_ids_padded)[0]
    predicted_tags = [idx2tag[np.argmax(tag)] for tag in predictions][:len(tokens)]
    return list(zip(tokens, predicted_tags))

In [28]:
# Example usage
print(pos_tag_sentence("ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवूच घेना ."))

1/1 [==============================] - 1s 1s/step
[('ग्रेटर', 'N_NN'), ('नोयडा', 'N_NN'), ('वेस्टाचे', 'N_NN'), ('त्रास', 'N_NN'), ('कमी', 'QT_QTF'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VF'), ('.', 'RD_PUNC')]


In [29]:
df_anwani = pd.read_excel('ref_sentences/anwani_ref_sentences.xlsx', engine='openpyxl')
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_anwani = df_anwani.iloc[:, :3]
df_anwani = df_anwani.iloc[:100, :3]
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...


In [31]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(pos_tag_sentence)
df_anwani.head()

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 1s 1s/step


,sentences_cleaned,predicted_sentences,tagged_sentences,output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,"[(आयकलां, V_VM_VF), (?, RD_PUNC), (लागीं, N_NS..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,"[(दोन, QT_QTC), (तीन, QT_QTC), (दिसांनी, N_NN)..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,"[(आनी, CC_CCD), (जाली, V_VM_VNF), (ना, RP_NEG)..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,"[(हांव, PR_PRP), (अशें, DM_DMD), (आसा, V_VM_VF..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,"[(पूण, CC_CCS), (हांगा, N_NST), (रावनय, N_NN),..."


In [32]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [33]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...


In [34]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

[48, 49]

In [35]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 98 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_cleaned     98 non-null     object
 1   tagged_sentences      98 non-null     object
 2   joined_output         98 non-null     object
 3   ref_sentences_length  98 non-null     int64 
 4   joined_sent_length    98 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.6+ KB


In [36]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [37]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,14,14,0.785714,0.714286,0.785714,0.738095,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, RD_PUNC, N_NST, RP_NEG, PR_PRP, N_NN..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.818182,0.818182,0.818182,0.818182,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VNF, JJ, N_NN, RD_..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,22,22,1.000000,1.000000,1.000000,1.000000,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,17,17,0.823529,0.852941,0.823529,0.835294,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...,15,15,0.866667,0.966667,0.866667,0.897778,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCS, N_NST, N_NN, V_VM_VNF, RP_NEG, RD_PUN..."


In [38]:
df_anwani.to_excel("test_gru_100_anwani.xlsx", index=False)

In [39]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        13
      CC_CCS       0.67      0.93      0.78        43
      DM_DMD       0.87      0.87      0.87        52
      DM_DMI       1.00      0.70      0.82        10
      DM_DMQ       0.80      0.27      0.40        15
          JJ       0.40      0.67      0.50        12
         NST       0.00      0.00      0.00         1
        N_NN       0.74      0.97      0.84       201
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.91      0.96      0.94        78
      PR_PRF       1.00      1.00      1.00         1
      PR_PRI       1.00      0.45      0.62        11
      PR_PRL       0.00      0.00      0.00         0
      PR_PRP       0.95      0.93      0.94       131
      PR_PRQ       0.62      0.75      0.68        20
         PSP       0.88      0.58      0.70        38
      QT_QTC       0.94      1.00      0.97        16
  

In [ ]:
ref_sentences = df_ref['ref_sentence'].tolist()
print(len(ref_sentences))
ref_sentences = ref_sentences[:75]
print(len(ref_sentences))

In [ ]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

In [ ]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df_test["sentence"], df_test["labels"] = zip(*df_test["actual_tags"].map(preprocess_sentence))
df_test.info()

In [ ]:
df_test.head()

In [ ]:
df_test['sentence'][0]

In [ ]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence

In [ ]:
df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

In [ ]:
df_test["output"] = df_test["joined_sent"].apply(pos_tag_sentence)
df_test.head()

In [ ]:
df_test.to_excel("test_biRNN_1_lakh_pos_tagger.xlsx", index=False)